# 🏆 Interview Questions — DSA

*Deep-dive Q&A with model answers.*

---
## 🏆 Interview Questions — DSA in Production APIs — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# DSA in Production APIs — Interview Questions

> Format: 5 architectural questions with deep-dive answers, then a multiple-choice
> knowledge check with an answer key, then a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. An endpoint that was fast in staging times out in production. It does `if item in collection` in a loop. What happened and how do you fix it?


**Deep dive.** In staging the collection was small, so the O(n) linear scan of a
`list` was invisible. In production the collection grew (say 50k rows), and
because the loop runs the scan once per requested item (m of them), the endpoint
is O(n·m). At 50k × 500 that's 25M comparisons *per request*, executed
synchronously — it saturates CPU and, in an async app, blocks the event loop so
*all* concurrent requests stall.

The fix is choosing the right structure: a `set`/`frozenset` gives O(1) average
membership, dropping the endpoint to O(n + m). The index (set) should be built
**once** (at startup or cached), not rebuilt per request. The deeper lesson is
that algorithmic complexity is a *production latency budget*: always ask "what is
n at scale, and what bounds it?" A structure that's fine at n=100 can be an
outage at n=10⁶.

---

### Q2. How does an unbounded request body become a denial-of-service vector, even without malicious intent?


**Deep dive.** If the endpoint accepts `list[str]` with no cap, the client
controls `m`. Combine that with an O(n) operation per element and the server-side
cost scales as O(n·m) under client control — an **algorithmic complexity attack**
(amplification). Even a well-meaning client running a large batch can trigger it.
The defense is to bound N at the trust boundary (Pydantic `max_length`), so the
worst-case cost is provably capped. Bounds on input size, page size, and nesting
depth are architectural decisions, not afterthoughts. Related: unbounded response
sizes (returning all rows) are the mirror problem — always paginate.

---

### Q3. When is an O(n²) algorithm inside an endpoint acceptable?


**Deep dive.** When N is small *and provably bounded*. If the input is a config
list capped at 50 items, an O(n²) nested loop is ~2,500 operations — negligible,
and often more readable than a cleverer structure. Premature optimization adds
risk (bugs, complexity) for no measurable gain. The senior signal is refusing to
optimize *or* to leave it slow without first establishing the bound on N and the
latency budget. "It depends on what bounds N" is the correct opening, not a
reflexive "always use a hash map."

---

### Q4. You need an autocomplete endpoint. Compare scanning a list, a SQL `LIKE`, and a trie.


**Deep dive.** *List scan / `startswith` over all rows* is O(n) per keystroke —
fine for thousands, hopeless for millions and it repeats every keystroke.
*SQL `LIKE 'prefix%'`* can use a B-tree index (prefix matches are
range scans) and is a reasonable default that offloads work to the database.
*Trie* gives O(k) prefix lookup independent of the number of keys (k = prefix
length), ideal for very high-QPS autocomplete, at the cost of memory and having
to keep the trie in sync with the source of truth. The choice is a trade-off of
QPS, dataset size, memory, and operational simplicity — for most apps a
DB-indexed `LIKE` or a search engine (Elasticsearch) beats hand-rolling a trie.

---

### Q5. Why is caching a computed result in a `dict` both a performance win and a correctness risk?


**Deep dive.** A `dict` (or Redis) cache converts an O(n) or O(log n) lookup into
O(1), which is a large win for read-heavy endpoints. But a cache is a **second
source of truth**, so it introduces invalidation — the hard problem. Risks: stale
entries when the source changes, unbounded memory growth without an eviction
policy (use an LRU/TTL), and the **thundering herd / cache stampede** when a hot
key expires and many requests recompute it at once. Senior caching always
specifies capacity, eviction, TTL, and a stampede mitigation (single-flight lock
or probabilistic early expiry) — not just "add a dict."

---

## Part 2 — Multiple-Choice Knowledge Check

**1. What is the time complexity of `x in my_list` for a Python `list`?**
- A) O(1)
- B) O(log n)
- C) O(n)
- D) O(n log n)

**2. Doing a list-membership test for `m` items against a list of `n` items is:**
- A) O(n + m)
- B) O(n · m)
- C) O(m log n)
- D) O(1)

**3. The best fix to make repeated membership tests O(1) average is to use a:**
- A) sorted list + linear scan
- B) `set` / `frozenset`
- C) tuple
- D) generator

**4. Accepting an unbounded `list` in a request body primarily risks:**
- A) a SQL injection
- B) an algorithmic-complexity denial of service
- C) a CSRF attack
- D) a memory leak in the client

**5. An O(n²) algorithm inside an endpoint is acceptable when:**
- A) never — always optimize
- B) the input size N is small and provably bounded
- C) only in staging
- D) the endpoint is a GET

### Answer Key
1. **C** — list membership is a linear scan.
2. **B** — m scans × O(n) each = O(n·m).
3. **B** — a set gives O(1) average membership.
4. **B** — unbounded N + per-element cost = algorithmic DoS.
5. **B** — a small, bounded N makes O(n²) negligible and often clearer.

---

## Part 3 — Gotchas Checklist

- **`in` on a list is O(n).** Use a `set`/`dict` for membership and keyed lookup.
- **Build the index once.** Rebuilding a set/dict per request throws away the win.
- **Bound every input.** `max_length` on lists, page-size caps, nesting limits —
  answer "what bounds N?" at the boundary.
- **Bound every output.** Returning all rows is the mirror DoS; paginate.
- **Hash-map worst case is O(n).** "O(1)" is *amortized/average*; pathological
  collisions (or untrusted keys) degrade it — usually fine, but know it exists.
- **Recursion depth ∝ input** is a crash vector (Python ~1000-frame limit); use
  iterative traversal on untrusted/deep data.
- **CPU-bound loops block the event loop** in `async` routes — offload or bound.
- **A cache is a second source of truth**: set capacity, eviction, TTL, and a
  stampede guard, or you trade a speed bug for a correctness bug.

---
## Scenario-Based Code Questions -- DSA

### Scenario 1 -- LRU Cache (O(1) get and put)

**Context:** ShopFlow caches rendered product HTML. On capacity, evict least-recently-used. Implement with O(1) for both operations using a doubly-linked list + hash map. **Do NOT use OrderedDict.**

```
cache = LRUCache(2)
cache.put(1, 'a'); cache.put(2, 'b')
cache.get(1)        # 'a' -- now MRU
cache.put(3, 'c')   # evicts key 2
cache.get(2)        # -1 (evicted)
```

In [ ]:
# -- SOLUTION --
class LRUCache:
    class Node:
        __slots__ = ('key', 'val', 'prev', 'next')
        def __init__(self, key=0, val=0):
            self.key, self.val, self.prev, self.next = key, val, None, None

    def __init__(self, capacity):
        self.cap, self.map = capacity, {}
        self.head, self.tail = self.Node(), self.Node()
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, n): n.prev.next, n.next.prev = n.next, n.prev

    def _push_front(self, n):
        n.prev, n.next = self.head, self.head.next
        self.head.next.prev = n; self.head.next = n

    def get(self, key):
        if key not in self.map: return -1
        n = self.map[key]; self._remove(n); self._push_front(n)
        return n.val

    def put(self, key, val):
        if key in self.map: self._remove(self.map[key])
        n = self.Node(key, val); self.map[key] = n; self._push_front(n)
        if len(self.map) > self.cap:
            lru = self.tail.prev; self._remove(lru); del self.map[lru.key]

cache = LRUCache(2)
cache.put(1, 'product_a'); cache.put(2, 'product_b')
assert cache.get(1) == 'product_a'
cache.put(3, 'product_c')
assert cache.get(2) == -1
assert cache.get(3) == 'product_c'
print('LRU Cache: all assertions passed |/')

### Scenario 2 -- Sliding Window Rate Limiter

**Context:** BuildFast limits each user to N requests per T-second sliding window. Unlike fixed windows, a sliding window counts requests in the ACTUAL last T seconds.

```
rl = SlidingWindowRateLimiter(3, 60)
rl.allow('u1', 0)   # True
rl.allow('u1', 65)  # True (t=0 expired)
```

In [ ]:
# -- SOLUTION --
from collections import defaultdict, deque

class SlidingWindowRateLimiter:
    def __init__(self, max_requests, window_seconds):
        self.max, self.window = max_requests, window_seconds
        self._log: dict = defaultdict(deque)

    def allow(self, user_id: str, timestamp: float) -> bool:
        dq = self._log[user_id]
        cutoff = timestamp - self.window
        while dq and dq[0] <= cutoff: dq.popleft()
        if len(dq) < self.max:
            dq.append(timestamp); return True
        return False

rl = SlidingWindowRateLimiter(3, 60)
results = [rl.allow('u1', t) for t in [0, 20, 40, 50, 65]]
print('allow() results:', results)
assert results == [True, True, True, False, True]
print('Sliding window rate limiter correct |/')

### Scenario 3 -- Pipeline Topological Sort + Cycle Detection

**Context:** BuildFast CI/CD jobs have dependencies. Detect circular dependencies and return valid execution order. Use Kahn's BFS (O(V+E)).

In [ ]:
# -- SOLUTION --

def validate_pipeline(jobs: dict):
    in_deg = {j: 0 for j in jobs}
    for job, deps in jobs.items():
        for dep in deps:
            in_deg[job] += 1
            if dep not in in_deg: in_deg[dep] = 0
    queue = deque(j for j, d in in_deg.items() if d == 0)
    order = []
    while queue:
        job = queue.popleft(); order.append(job)
        for nj, deps in jobs.items():
            if job in deps:
                in_deg[nj] -= 1
                if in_deg[nj] == 0: queue.append(nj)
    return order if len(order) == len(in_deg) else None

jobs = {'checkout':[],'install':['checkout'],'test':['install'],'build':['test'],'deploy':['build']}
print('Valid order:', validate_pipeline(jobs))
cyclic = {'a':['c'],'b':['a'],'c':['b']}
print('Cyclic pipeline:', validate_pipeline(cyclic))  # None

### Scenario 4 -- Consistent Hashing Ring

**Context:** ShopFlow distributes product cache across 3 nodes. When a node is removed, only ~1/3 of keys should be remapped (not all).

In [ ]:
# -- SOLUTION --
import hashlib
import bisect

class ConsistentHashRing:
    def __init__(self, replicas=100):
        self.replicas, self._ring, self._nodes = replicas, [], {}

    def _hash(self, key): return int(hashlib.md5(key.encode()).hexdigest(), 16)

    def add_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f'{node}#{i}')
            self._nodes[h] = node; bisect.insort(self._ring, h)

    def remove_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f'{node}#{i}')
            del self._nodes[h]; self._ring.pop(bisect.bisect_left(self._ring, h))

    def get_node(self, key):
        if not self._ring: return None
        idx = bisect.bisect_right(self._ring, self._hash(key)) % len(self._ring)
        return self._nodes[self._ring[idx]]

ring = ConsistentHashRing(50)
for n in ['cache-1','cache-2','cache-3']: ring.add_node(n)
keys = [f'user:{i}' for i in range(1000)]
before = {k: ring.get_node(k) for k in keys}
ring.remove_node('cache-2')
after = {k: ring.get_node(k) for k in keys}
remapped = sum(1 for k in keys if before[k] != after[k])
print(f'Remapped: {remapped}/1000 ({remapped/10:.1f}%) -- expect ~33%')